## Medications
Notes: [WHO ATC Codes](https://atcddd.fhi.no/atc_ddd_index/)

Some background information:
# ATC code
The ATC is a structured, hierarchical code. The code start generically and becomes increasingly more detailed.
The ATC structure is described as follows, take an example ```A02BC01```
- *A*: ATC 1st level -> **Main group**, in this case *Alimentary tract and metabolism*
- *02*: ATC 2nd level -> **Therapeutic group**, in this case *Drugs for acid-related disorders*
- *B*: ATC 3th level -> **Pharmacological group**, in this case *Drugs for peptic ulcer and gastro-oesophageal reflux disease*
- *C*: ATC 4th level -> **Chemical group**, in this case *Proton pump inhibitors*
- *01*: ATC 5th level -> **Active substance**, in this case *Omeprazole*

Main groups:
```
Alimentary tract and metabolism (A)
Blood and blood forming organs (B)
Cardiovascular system (C)
Dermatologicals (D)
Genito urinary system and sex hormones (G)
Systemic hormonal preparations, excl. sex hormones and insulins (H)
Antiinfectives for systemic use (J)
Antineoplastic and immunomodulating agents (L)
Musculo-skeletal system (M)
Nervous system (N)
Antiparasitic products, insecticides and repellents (P)
Respiratory system (R)
Sensory organs (S)
Various (V)
```

You can find ATC/CUI/SNOMED mappings here: ```T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS```

# The dataset contains more information than just the medication

* ```Med_Mednaam```: full name of medication including dosage
* ```category_display_original```: 
  * ```Opname```: during admission to hospital -> more acute situation
  * ```Thuis/verpleeghuis```: during stay in (care) home -> less acute situation
* ```dosInstr_doseQuantity_unit```: unit of dosage
* ```dosInstr_doseQuantity_value```: dosage value (measured in the unit of dosage)
* ```dosInstr_frequency_value```: frequency (per day)
* ```dosInstr_frequency_unit```: frequency unit (day/month/year)
* ```dosInstr_PerBetwDos_value```: number of time units between between dosage intake-sessions
* ```dosInstr_PerBetwDosUnit_Orig```: time unit definition
* ```intendedRoute_display```: method in of ingestion
* ```status_display_original```: first recipe/continuation/stop
* ```usageDuration_valueDuration_valu```: prescribed duration
  
To interpret the timing features, suppose;
* ```dosInstr_frequency_value = 4```
* ```dosInstr_frequency_unit = 'day'```
* ```dosInstr_PerBetwDos_value = 14```
* ```dosInstr_PerBetwDosUnit_Orig = 'day'```
Means: once per $14$ days the patient takes the medication $4$ times in a day.

If ```dosInstr_PerBetwDos_value``` is ```None``` then the medication is simply taken ```dosInstr_frequency_value``` per ```dosInstr_frequency_unit```.

# The problem: high cardinality

In our case we have about $900$ unique ATC codes. This is too high for any practical purpose, also there 
will be a lot of sparsity, most of the ATC values will be $0$ (assuming $1$ for presence, and $0$ for absence).

## Solutions and suggestions

### Make use of the hierarchical nature
The cardinality will increase if we add more ATC levels. Perhaps ATC level1+level2+level3 is enough and would allow basic one-hot encoding.
In fact, we can make feature combinations: ```level1+level2+level3, level1+level2, level1+level3, level2+level3```.

### Filter out the non-informative features

### One-hot encoding followed by dimension reduction

### Express each ATC code with a pre-trained embedding
* An aggregation of word embeddings: ```T:\laupodteam\AIOS\SUPPORTING_DATA\WordVectors```
* Document embeddings using SBERT: ```T:\laupodteam\AIOS\SUPPORTING_DATA\SentenceEmbedders```
* Graph embeddings: ```T:\laupodteam\AIOS\SUPPORTING_DATA\OntologyEmbeddings```
* A combination of the above

To make use of document/sentence embeddings as completely as possible maybe combine
```Med_Mednaam```, ```category_display_original```, ```intendedRoute_display```, ```status_display_original``` into a short narrative.

Also, perhaps make use of the 


### Use supervised feature encoding
* [NCA](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html)/[LDA](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html)
* [WoE](https://feature-engine.trainindata.com/en/latest/user_guide/encoding/WoEEncoder.html)

In [7]:
import pandas as pd
import os
import gc

input_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805'
output_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data'

In [8]:
ATC_CUI = pd.read_parquet(r'T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS\ATC_CUI.parquet')
CUI_DEF = pd.read_parquet(r'T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS\CUI_DEF.parquet')
ATC_STR = CUI_DEF.loc[CUI_DEF.LAT=='ENG'].merge(ATC_CUI, on='CUI', how='inner')
ATC_STR = ATC_STR.assign(STR=ATC_STR.STR.str.lower())
ATC_STR = ATC_STR[~ATC_STR.duplicated(subset=['CUI', 'ATC_code'])]

In [ ]:
ATC_STR.STR.unique().shape

In [4]:
CUI_CUI = pd.read_parquet(r'T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS\CUI_CUI.parquet')

In [1]:
import pandas as pd
import os

output_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data'
os.chdir(output_data)

In [ ]:
# Caricamento del dataset
df_medicatie = pd.read_parquet(r'parquet\df_medicatie.parquet')

# Eventuale rimozione dei duplicati
df_medicatie.drop_duplicates(keep='first', inplace=True)

# Analisi per capire quali colonne hanno una percentuale di NaN
# inferiore ad una certa soglia
missing_percentage = df_medicatie.isnull().mean()
keep_cols = df_medicatie.loc[:, missing_percentage < 0.2].columns

print(f'{keep_cols}, # = {len(keep_cols)}\n')
print(f'Total number of records: {df_medicatie.shape[0]}\n')
print(f'Number of unique patients: {df_medicatie["studyId_0831"].nunique()}\n')
print(f'Number of missing ATC codes: {len(df_medicatie[df_medicatie["code5_ATC_code"].isna()].code5_ATC_code)}')

In [30]:
# Viene caricato il dizionario per gli ATC codes
dictionary = pd.read_csv('T:\laupodteam\AIOS\SUPPORTING_DATA\ATC\WHO ATC-DDD 2024-09-25.csv',sep='\t')

# Trova tutte le righe con NaN nella colonna specificata
missing_ATC = df_medicatie[df_medicatie['code5_ATC_code'].isna() | \
    df_medicatie['code5_ATC_display'].isna()].code2_HPK_display

# Rimozione di tutti i record il cui ATC code è mancante
df_medicatie = df_medicatie.dropna(subset=['code5_ATC_code', 'code5_ATC_display'])

# Inizializzazione della lista contenente le stringhe degli ATC codes
ATC_strings = [] 

for code in df_medicatie['code5_ATC_code']:
    # Inizializzare la stringa con 'Main group'
    string = 'Main group: ' 
    
    # Estrarre i valori dalle query e concatenarli, gestendo i risultati vuoti
    main_group = dictionary.query(f'atc_code == "{code[0]}"')
    if not main_group.empty:
        string += main_group.atc_name.values[0] + '. '
    
    therapeutic_group = dictionary.query(f'atc_code == "{code[0:3]}"')
    if not therapeutic_group.empty:
        string += 'Therapeutic group: ' + therapeutic_group.atc_name.values[0] + '. '
    
    pharmacological_group = dictionary.query(f'atc_code == "{code[0:4]}"')
    if not pharmacological_group.empty:
        string += 'Pharmacological group: ' + pharmacological_group.atc_name.values[0] + '. '
    
    chemical_group = dictionary.query(f'atc_code == "{code[0:5]}"')
    if not chemical_group.empty:
        string += 'Chemical group: ' + chemical_group.atc_name.values[0] + '. '
    
    active_substance = dictionary.query(f'atc_code == "{code}"')
    if not active_substance.empty:
        string += 'Active substance: ' + active_substance.atc_name.values[0]

    # Aggiungere la stringa alla lista
    ATC_strings.append(string)


""" for code in df_medicatie['code5_ATC_code']:
    string = str('Main group: ') + dictionary.query(f'atc_code == "{code[0]}"').atc_name.values[0] + str('. ') + \
        str('Therapeutic group: ') + dictionary.query(f'atc_code == "{code[0:3]}"').atc_name.values[0] + str('. ') + \
        str('Pharmacologial group: ') + dictionary.query(f'atc_code == "{code[0:4]}"').atc_name.values[0] + str('. ') + \
        str('Chemical group: ') + dictionary.query(f'atc_code == "{code[0:5]}"').atc_name.values[0] + str('. ') + \
        str('Active substance: ') + dictionary.query(f'atc_code == "{code}"').atc_name.values[0]
    
    ATC_strings.append(string) """

# La lista contenente le stringhe degli ATC codes viene inserita all'interno del dataframe
df_medicatie.insert(df_medicatie.columns.get_loc('code5_ATC_code') + 1, 'ATC_strings', ATC_strings)

# Salvo il dataframe nel percorso 'output_data'
df_medicatie.to_parquet('df_medicatie_ATC.parquet')

## Start from here (30/09/2024)

In [5]:
import pandas as pd


df_medicatie = pd.read_parquet(r'C:\Users\ddellamo\Desktop\df_medicatie_ATC.parquet')

str

Below insert medications with hierarchical code:
- How many A,B,C...?
- How many subgroup..?

Below insert medications with count

In [9]:
#df_medicatie_count = df_medicatie.drop_duplicates().sort_index().groupby('studyId_0831').agg(lambda x: len(x.to_list()))
#df_medicatie_count

In [10]:
#df_medicatie_count.reset_index().to_csv(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\csv\df_medicatie.csv', index=False)

In [9]:
C = "A01AB09"

In [ ]:
final_str = ''

In [ ]:
final_str += prova.query(f'atc_code == "{C[0]}"').atc_name + \
            prova.query(f'atc_code == "{C[0:3]}"').atc_name + \
            prova.query(f'atc_code == "{C[0:3]}"').atc_name
    